In [1]:
from config_el import *

In [2]:
data = 'D:\Documents\github\diabetes_prediction\diabetes_prediction_system\data_set\diabetes_cleanned_data.csv'

df = pd.read_csv(data)
df.head(2)

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148.0,72.0,35.0,169.0,33.6,0.627,50,1
1,1,85.0,66.0,29.0,58.6,26.6,0.351,31,0


In [3]:
X = df.drop('Outcome',axis=1)
y = df['Outcome']

In [4]:
X_train,X_test,y_train,y_test = train_test_split(X, y, test_size=0.20, random_state=42)

In [5]:
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features="sqrt",
    bootstrap=True,
    class_weight=None,
    random_state=42
)

In [6]:
# fitting the model to learn

rf.fit(X_train,y_train)

RandomForestClassifier(random_state=42)

In [7]:
y_pred =rf.predict(X_test)

y_pred

array([1, 0, 0, 0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0,
       0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0,
       0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 1,
       0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0,
       0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 1,
       0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1,
       0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0])

In [9]:
# check accuracy_score, classification_report,confusion_matrix

print('accuracy_score',accuracy_score(y_test,y_pred))

print('classification_report',classification_report(y_test,y_pred))

print('confusion_matrix')
print(confusion_matrix(y_test,y_pred))

accuracy_score 0.7337662337662337
classification_report               precision    recall  f1-score   support

           0       0.82      0.76      0.79        99
           1       0.61      0.69      0.65        55

    accuracy                           0.73       154
   macro avg       0.71      0.72      0.72       154
weighted avg       0.74      0.73      0.74       154

confusion_matrix
[[75 24]
 [17 38]]


confusion matrix is not satisifed so moved to the grid search cv modelling

In [10]:
from sklearn.model_selection import GridSearchCV

In [12]:
param_grid = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [5, 10, 15, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2'],
    'bootstrap': [True],
    'class_weight': [None, 'balanced']
}

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    scoring='recall',
    cv=5,
    n_jobs=-1,
    verbose=2
)


In [13]:
# fitting the data to the grid_search model

grid_search.fit(X_train,y_train)

print('best parameter')
print(grid_search.best_params_)

# recall

print('recall',grid_search.best_score_)

best_rf  = grid_search.best_estimator_

Fitting 5 folds for each of 576 candidates, totalling 2880 fits
best parameter
{'bootstrap': True, 'class_weight': 'balanced', 'max_depth': 5, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 10, 'n_estimators': 100}
recall 0.7841638981173864


In [14]:
gs_pred = best_rf.predict(X_test)

In [15]:
print('accuracy_score',accuracy_score(y_test,gs_pred))
print('classification_report')
print(classification_report(y_test,gs_pred))
print('confusion_matrix',confusion_matrix(y_test,gs_pred))

accuracy_score 0.7532467532467533
classification_report
              precision    recall  f1-score   support

           0       0.89      0.71      0.79        99
           1       0.61      0.84      0.71        55

    accuracy                           0.75       154
   macro avg       0.75      0.77      0.75       154
weighted avg       0.79      0.75      0.76       154

confusion_matrix [[70 29]
 [ 9 46]]


In [20]:
# saving the model 

import joblib

final_model_estimator = grid_search.best_estimator_
final_model_params = grid_search.best_params_


final_model = joblib.dump(grid_search, 'diabetes_predicting_model.joblib')

In [22]:
# loading the 'diabetes_predicting_model.joblib'

load_model  = joblib.load('diabetes_predicting_model.joblib')

In [35]:
# predicting the values

load_model_pred = load_model.predict([[  6.   , 148.   ,  72.   ,  35.   , 169.   ,  33.6  ,   0.627, 50]])

c:\Users\manoj\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


In [36]:
load_model_pred

array([1])

In [34]:
input = df.head(1)
to_arr = input.to_numpy()

to_arr[0:5]

array([[  6.   , 148.   ,  72.   ,  35.   , 169.   ,  33.6  ,   0.627,
         50.   ,   1.   ]])

**grind search cv is performing well**
Out of every 100 diabetic patients, your model correctly identifies about 84.

That's a strong improvement from the earlier models.

 then for more performance moving to the randomsearchcv modelling

In [ ]:
from sklearn.model_selection import RandomizedSearchCV


# Parameter Distribution
param_dist = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [5, 10, 15, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2'],
    'bootstrap': [True],
    'class_weight': [None, 'balanced']
}

# Random Search
random_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_distributions=param_dist,
    n_iter=50,
    scoring='recall',
    cv=5,
    random_state=42,
    n_jobs=4,          # better for 8GM ram
    verbose=2
)


In [17]:
random_search.fit(X_train,y_train)

Fitting 5 folds for each of 50 candidates, totalling 250 fits


RandomizedSearchCV(cv=5, estimator=RandomForestClassifier(random_state=42),
                   n_iter=50, n_jobs=4,
                   param_distributions={'bootstrap': [True],
                                        'class_weight': [None, 'balanced'],
                                        'max_depth': [5, 10, 15, None],
                                        'max_features': ['sqrt', 'log2'],
                                        'min_samples_leaf': [1, 2, 4],
                                        'min_samples_split': [2, 5, 10],
                                        'n_estimators': [100, 200, 300, 500]},
                   random_state=42, scoring='recall', verbose=2)

In [ ]:
print('best score',random_search.best_score_)

best score 0.774750830564784


In [19]:
# best model

bestrs_rf = random_search.best_estimator_


In [21]:
# predict

y_pred_rs = bestrs_rf.predict(X_test)

In [24]:
print('accuracy_score',accuracy_score(y_test,y_pred_rs))

print('classification_report',classification_report(y_test,y_pred_rs))
print('confusion_matrix')
print(confusion_matrix(y_test,y_pred_rs))

accuracy_score 0.7467532467532467
classification_report               precision    recall  f1-score   support

           0       0.88      0.70      0.78        99
           1       0.61      0.84      0.70        55

    accuracy                           0.75       154
   macro avg       0.74      0.77      0.74       154
weighted avg       0.78      0.75      0.75       154

confusion_matrix
[[69 30]
 [ 9 46]]


**Conclusion**

grindSearch CV

accuracy_score 0.7532467532467533
classification_report
              precision    recall  f1-score   support

           0       0.89      0.71      0.79        99
           1       0.61      0.84      0.71        55

    accuracy                           0.75       154
   macro avg       0.75      0.77      0.75       154
weighted avg       0.79      0.75      0.76       154

confusion_matrix: [[70 29]
                  [ 9 46]]

randomSrearchCV

accuracy_score 0.7467532467532467
classification_report               precision    recall  f1-score   support

           0       0.88      0.70      0.78        99
           1       0.61      0.84      0.70        55

    accuracy                           0.75       154
   macro avg       0.74      0.77      0.74       154
weighted avg       0.78      0.75      0.75       154

confusion_matrix:
                  [[69 30]
                  [ 9 46]]


**GrindSearch CV** and **RandomSearch CV** are performing giving similar **accuracy** and also giving similar **confusion matrix** so we can use one of them